# Validate Gold Layer
Validates the accuracy and consistency of Gold Layer metrics against the underlying Silver Layer data.

In [0]:
SELECT *
FROM ecommerce_catalog.gold.daily_sales

In [0]:
SELECT COUNT(DISTINCT sales_date) AS total_days, COUNT(sales_date) AS total_rows
FROM ecommerce_catalog.gold.daily_sales;

In [0]:
SELECT 
    COUNT(CASE WHEN sales_date IS NULL THEN 1 END)      AS nulls_in_sales_date,
    COUNT(CASE WHEN order_count IS NULL THEN 1 END)     AS nulls_in_order_count,
    COUNT(CASE WHEN units_sold IS NULL THEN 1 END)      AS nulls_in_units_sold,
    COUNT(CASE WHEN revenue IS NULL THEN 1 END)         AS nulls_in_revenue,
    COUNT(CASE WHEN unique_customers IS NULL THEN 1 END)AS nulls_in_unique_customers,
    COUNT(CASE WHEN order_count IS NULL THEN 1 END)     AS nulls_in_order_count,
    COUNT(CASE WHEN avg_order_value IS NULL THEN 1 END) AS nulls_in_avg_order_value
FROM ecommerce_catalog.gold.daily_sales;

In [0]:
SELECT
    COUNT(CASE WHEN order_count < 0 THEN 1 END)     AS negative_order_count,
    COUNT(CASE WHEN units_sold < 0 THEN 1 END)      AS negative_units_sold,
    COUNT(CASE WHEN revenue < 0 THEN 1 END)         AS negative_revenue,
    COUNT(CASE WHEN unique_customers < 0 THEN 1 END)AS negative_unique_customers,
    COUNT(CASE WHEN order_count < 0 THEN 1 END)     AS negative_order_count,
    COUNT(CASE WHEN avg_order_value < 0 THEN 1 END) AS negative_avg_order_value    
FROM ecommerce_catalog.gold.daily_sales;



In [0]:
SELECT MIN(sales_date) AS min_date, MAX(sales_date) AS max_date
FROM ecommerce_catalog.gold.daily_sales;

In [0]:
SELECT 
    (SELECT ROUND(SUM(revenue), 2) FROM ecommerce_catalog.gold.daily_sales) 
    = 
    (SELECT ROUND(SUM(quantity * unit_price), 2) FROM ecommerce_catalog.silver.silver_orders) 
    AS is_matching;

In [0]:
SELECT 
    (SELECT (SUM(order_count)) FROM ecommerce_catalog.gold.daily_sales) 
    = 
    (SELECT (COUNT(order_id)) FROM ecommerce_catalog.silver.silver_orders) 
    AS is_matching;

In [0]:
SELECT COUNT(DISTINCT product_id) AS distinct_product_id, COUNT(product_id) AS total_product_id
FROM ecommerce_catalog.gold.product_performance;

In [0]:
SELECT 
    (SELECT ROUND(SUM(revenue), 2) FROM ecommerce_catalog.gold.product_performance) 
    = 
    (SELECT ROUND(SUM(revenue), 2) FROM ecommerce_catalog.gold.daily_sales)
    AS is_matching,
     
    (SELECT ROUND(SUM(revenue), 2) FROM ecommerce_catalog.gold.daily_sales)
    =
    (SELECT ROUND(SUM(quantity * unit_price), 2) FROM ecommerce_catalog.silver.silver_orders)
    AS is_matching_too;   

In [0]:
SELECT 
    (SELECT (SUM(units_sold)) FROM ecommerce_catalog.gold.product_performance) 
    = 
    (SELECT (SUM(quantity)) FROM ecommerce_catalog.silver.silver_orders)
    AS is_matching;

In [0]:
SELECT COUNT(DISTINCT customer_id) as distinct_customer_id, COUNT(customer_id) as total_customer_id
FROM ecommerce_catalog.gold.customer_metrics;


In [0]:
SELECT 
    (SELECT ROUND(SUM(total_revenue), 2) FROM ecommerce_catalog.gold.customer_metrics) 
    = 
    (SELECT ROUND(SUM(unit_price * quantity), 2) FROM ecommerce_catalog.silver.silver_orders)
    AS is_matching;

In [0]:
SELECT 
    (SELECT ROUND(SUM(unit_price * quantity), 2) FROM ecommerce_catalog.silver.silver_orders
    WHERE customer_id IS NOT NULL)
    AS total_revenue_silver,
    (SELECT ROUND(SUM(total_revenue), 2) FROM ecommerce_catalog.gold.customer_metrics)
    AS total_revenue_gold;

### **Total Revenue Discrepancy:**
The total revenue aggregated from `customer_metrics` is lower than the total revenue from `silver.orders`. The difference is caused by orders with missing `customer_id` values, which cannot be attributed to any row in `customer_metrics`.

A comparison with `silver.orders` excluding `customer_id IS NULL` confirms that the revenues match exactly. The discrepancy is therefore domain-explainable and does not represent a calculation error in the Gold layer.

In [0]:
SELECT
    (SELECT (SUM(total_units)) FROM ecommerce_catalog.gold.customer_metrics)
    AS total_units_gold,
    (SELECT (SUM(quantity)) FROM ecommerce_catalog.silver.silver_orders
    WHERE customer_id IS NOT NULL)
    AS total_units_silver;

In [0]:
SELECT
    COUNT(CASE WHEN first_order_date > last_order_date THEN 1 END)     AS wrong_dates,
    count(CASE WHEN order_count > 0 THEN 1 END)                        As total_customers  
FROM ecommerce_catalog.gold.customer_metrics;

In [0]:
SELECT * FROM ecommerce_catalog.gold.customer_metrics
WHERE customer_id = "C0478";

In [0]:
SELECT
    (SELECT COUNT(customer_id) FROM ecommerce_catalog.silver.silver_orders
    WHERE customer_id = "C0478") 
    AS selected_customer_orders,
    (SELECT SUM(quantity) FROM ecommerce_catalog.silver.silver_orders
    WHERE customer_id = "C0478") 
    AS selected_customer_units,
    (SELECT (ROUND(SUM(quantity * unit_price), 2)) FROM ecommerce_catalog.silver.silver_orders
    WHERE customer_id = "C0478") 
    AS selected_customer_revenue;

In [0]:
SELECT * 
FROM ecommerce_catalog.gold.conversion_funnel;

In [0]:
SELECT
    (SELECT COUNT(DISTINCT session_id) FROM ecommerce_catalog.silver.silver_clickstream
    WHERE event_type = "product_view")
    AS product_views,
    (SELECT COUNT(DISTINCT session_id) FROM ecommerce_catalog.silver.silver_clickstream
    WHERE event_type = "add_to_cart")
    AS add_to_cart,
    (SELECT COUNT(DISTINCT session_id) FROM ecommerce_catalog.silver.silver_clickstream
    WHERE event_type = "checkout_start")
    AS checkout_start,
    (SELECT COUNT(DISTINCT session_id) FROM ecommerce_catalog.silver.silver_clickstream
    WHERE event_type = "purchase")
    AS purchase;


In [0]:
SELECT 
    (SELECT ROUND(SUM(unit_price * quantity), 2) FROM ecommerce_catalog.silver.silver_orders)
    AS total_revenue_order_with_null,
    (SELECT ROUND(SUM(unit_price * quantity), 2) FROM ecommerce_catalog.silver.silver_orders
    WHERE customer_id IS NOT NULL)
    AS total_revenue_order_without_null,
    (SELECT ROUND(SUM(total_revenue), 2) FROM ecommerce_catalog.gold.customer_metrics)
    AS total_revenue_customer_metrics,
    (SELECT ROUND(SUM(revenue), 2) FROM ecommerce_catalog.gold.daily_sales)
    AS total_revenue_daily_sales,
    (SELECT ROUND(SUM(revenue), 2) FROM ecommerce_catalog.gold.product_performance)
    AS total_revenue_product_performance;


### **Revenue Reconciliation Across Gold Tables:**
`daily_sales` and `product_performance` contain the full revenue from `silver.orders`, including orders without a `customer_id`.
In contrast, `customer_metrics` only accounts for orders that could be attributed to a known customer. As a result, the total revenue in `customer_metrics` is lower.
A comparison with `silver.orders` excluding records where `customer_id IS NULL` confirms that the revenues match.